### ==============================================================================
## Processing of Moving Vessel Profiler Data - code 3_lag_correction
### Authors: Elisabet Verger-Miralles (everger@imedea.uib-csic.es)
### Data from FaSt-SWOT experiment
# 
**DESCRIPTION:**
This script performs Step 3: Application of the Lag Correction. 
It reads the linear regression models and regime classifications generated 
in Step 2. Based on the profile's specific regime (e.g., Leg 1 Normal, 
Leg 1 Anomaly, or Leg 2) and its average descent rate (w), it calculates 
the precise scan lag required. It then shifts the Temperature signal by 
interpolating it over the calculated lag and recalculates Practical Salinity.
It also includes an optional diagnostic plotting feature for specific profiles.
#
INPUT: QC NetCDF files (*_step1_qc.nc), lag models CSV, and regimes CSV.
#
OUTPUT: NetCDF files with lag-corrected Temperature and Salinity (*_step3_lag.nc).
### ==============================================================================


In [1]:
import xarray as xr
import numpy as np
import pandas as pd
import gsw
import matplotlib.pyplot as plt
from pathlib import Path
import re
import warnings

warnings.filterwarnings("ignore")

# ==========================================
# 1. CONFIGURATION
# ==========================================
BASE_ROOT = Path(r"C:\Users\ASUS\Desktop\MVP\MVP_paper_data_figures_publish\MVP_fastswot_nc_final_processing")
DIRS_IN = {
    "FaSt_Leg1": BASE_ROOT / "Data" / "Leg1" / "processed_step1_highres_qc",
    "FaSt_Leg2": BASE_ROOT / "Data" / "Leg2" / "processed_step1_highres_qc",
}
DIRS_OUT = {
    "FaSt_Leg1": BASE_ROOT / "Data" / "Leg1" / "processed_step3_lag",
    "FaSt_Leg2": BASE_ROOT / "Data" / "Leg2" / "processed_step3_lag",
}
# Create output directories if they don't exist
for d in DIRS_OUT.values(): d.mkdir(parents=True, exist_ok=True)

MODELS_CSV = BASE_ROOT / "Figures" / "STEP2_LAG_ANALYSIS_SEGMENTS" / "lag_models_step2.csv"
REGIMES_CSV = BASE_ROOT / "Figures" / "STEP2_LAG_ANALYSIS_SEGMENTS" / "profile_regimes.csv"

# Specific MVP IDs to plot for visual diagnostic checks
IDS_A_VISUALIZAR = ["0626", "0627", "0628"]

# ==========================================
# 2. FUNCTIONS
# ==========================================
def get_mvp_id(filename):
    """Extracts the 4-digit MVP profile ID from the filename."""
    match = re.search(r'_mvp_(\d{4})_', filename)
    return match.group(1) if match else None

def shift_signal_interp(data, lag_scans):
    """Shifts a 1D data array by a fractional number of scans using linear interpolation."""
    x = np.arange(len(data))
    return np.interp(x, x + lag_scans, data, left=np.nan, right=np.nan)

# def plot_diagnostic(p, t_raw, c_raw, s_raw, t_lag, s_lag, info):
#     """Generates a 3-panel plot comparing raw vs. lag-corrected profiles."""
#     fig, axs = plt.subplots(1, 3, figsize=(15, 6), sharey=True, constrained_layout=True)
#     tit_col = 'red' if 'Anomaly' in info['regime'] else 'blue'
    
#     axs[0].plot(t_raw, p, 'k-', lw=1, alpha=0.3, label='Raw')
#     axs[0].plot(t_lag, p, 'r--', lw=1.5, label='Lagged')
#     axs[0].set_title("Temperature [°C]")
#     axs[0].invert_yaxis()
#     axs[0].legend()
    
#     axs[1].plot(c_raw, p, 'k-', lw=1, alpha=0.5)
#     axs[1].set_title("Conductivity [mS/cm]")
    
#     axs[2].plot(s_raw, p, 'k-', lw=1, alpha=0.3, label='Raw Sal')
#     axs[2].plot(s_lag, p, 'b-', lw=1.5, label='Lag-Corrected')
#     axs[2].set_title("Practical Salinity")
#     axs[2].legend()
    
#     fig.suptitle(f"MVP {info['id']} | Regime: {info['regime']} | Lag: {info['lag']:.2f} scans", 
#                  fontsize=14, fontweight='bold', color=tit_col)
#     plt.show()

# ==========================================
# 3. MAIN PROCESSING ROUTINE
# ==========================================
# Load the linear regression models generated in Step 2
models_df = pd.read_csv(MODELS_CSV)
models = {row['regime']: {'m': row['slope'], 'c': row['intercept']} for _, row in models_df.iterrows()}
regime_map = pd.read_csv(REGIMES_CSV).set_index('file')['regime'].to_dict()

print(" Starting Step 3: Lag Application...")

for leg_name, input_dir in DIRS_IN.items():
    files = sorted(list(input_dir.glob("*.nc")))
    for f in files:
        try:
            file_id = get_mvp_id(f.name)
            # Retrieve the profile's regime. 
            # If not found, assume anomaly for Leg 1 as a precaution, standard for Leg 2.
            raw_regime = regime_map.get(f.name, "Leg1_Anomaly" if "Leg1" in leg_name else "Leg2_Standard")
            regime_key = raw_regime.replace("_", " ")
            
            # Select the appropriate lag model
            model = models.get(regime_key, models['FaSt-SWOT Leg1 Anomaly' if "Leg1" in leg_name else 'FaSt-SWOT Leg2'])
            
            with xr.open_dataset(f) as ds:
                p, t_raw, c_raw, w = ds.pressure.values, ds.t1.values, ds.c1.values, ds.w_descent.values
                
                # Calculate mean descent rate (w) for the profile's valid depth range
                mask_w = (p > 10) & (p < 200) & np.isfinite(w)
                w_mean = np.mean(w[mask_w]) if mask_w.sum() > 50 else np.nanmean(w)
                
                # Calculate the final lag in scans to apply
                lag_scans = model['m'] * w_mean + model['c']
                
                # Apply the shift to Temperature
                t_lagged = shift_signal_interp(t_raw, -lag_scans)
                # Calculate Salinity using the GSW library for both raw and lagged states
                s_raw = gsw.SP_from_C(c_raw, t_raw, p)
                s_lagged = gsw.SP_from_C(c_raw, t_lagged, p)
                
                # # Plot diagnostic figure if the ID is in the watch list
                # if file_id in IDS_A_VISUALIZAR:
                #     info = {'id': file_id, 'regime': regime_key, 'lag': lag_scans}
                #     plot_diagnostic(p, t_raw, c_raw, s_raw, t_lagged, s_lagged, info)

                # Save the corrected variables into a new NetCDF dataset
                ds_out = ds.copy(deep=True)
                ds_out['t1_lagged'] = (('scan',), t_lagged)
                ds_out['salinity_lagged'] = (('scan',), s_lagged)
                ds_out.to_netcdf(DIRS_OUT[leg_name] / f.name.replace("_step1_qc.nc", "_step3_lag.nc"))
                
        except Exception as e:
            print(f"❌ Error processing {f.name}: {e}")

print("Step 3 Completed. All profiles lag-corrected.")

c:\Users\ASUS\anaconda3\envs\env_elisabet\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


 Starting Step 3: Lag Application...
Step 3 Completed. All profiles lag-corrected.
